<a href="https://colab.research.google.com/github/Nick300500/Electricity-Price-Forecast-Battery-Optimization/blob/main/Battery_optimization_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Price Forecasting and Battery Scheduling

# General information

In this script, we tried to analyze, how much profit can accurate electricity price forecasting unlock when operating a residential battery system.

To do so, first we implemented a forecasting model.
Therefore, we set up a dataset with data we thought could be important for the Day-Ahead price.

Then we create a forecasting-model using RandomForest. WIth this model, we forecast a Day-Ahead Price and put it into a given battery optimizer to calculate the profit. As battery, we use the examplary home-storage battery (Sungrow SBR096 with 3 Module and the given data from the datasheet.

Afterwards we also compare the profit of the perfect battery scheduling with the profit we make, using the charging/discharging-behavior from the battery-optimazation, using our forecasted prices and the actual prices of the Day-Ahead.
Finally, we do a sensitivity-analysis to investigate the influence of the price-prediction on our profit.

In our code, we # some displays due to clarity. To visualize them, just remove the #

**Code structure:** the actual pipeline logic (optimizer, data loading/prep, modeling, backtesting, plotting) lives in the `src/battery_opt` package. This notebook only orchestrates: load -> prepare -> train -> backtest -> analyze, so each stage can be re-run independently instead of only top-to-bottom.

# Setup

In [ ]:
#install packages
!pip install pyomo
!pip install highs

In [ ]:
#make the battery_opt package importable (repo_root/src/battery_opt)
import sys
sys.path.insert(0, "src")

# On Windows, stdout can default to the system codepage instead of UTF-8,
# which mangles the euro sign in printed output.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import os

import pandas as pd

from battery_opt.config import DEFAULT_STORAGE_PARAMS, BATTERY_SYSTEM_COST_EUR, DATA_DIR, RESULTS_DIR, ensure_dirs
from battery_opt.optimizer import run_dispatch
from battery_opt import data_loading, data_prep, modeling, backtest, plotting, smard_client, entsoe_client

ensure_dirs()

# Storage optimization function (as provided)

First, we use the storage optimization function, which was provided to us (now in `battery_opt/optimizer.py`). This function takes a list of prices and optimizes the charging and discharging behavior of a battery (the characteristics of the battery are the default ones for our example, see `DEFAULT_STORAGE_PARAMS`). It gives an output which includes charging, discharging, Energy and profit per timestep (one hour).

To check, whether it is working correctly, we implement the scenario of perfect prediction, which means the operation of a battery concidering the actual Day-Ahead prices. To check the correct working, we plot also an exemplary timeframe and check, whether the charging happens at local minima and the discharging happens at local maxima.

In [ ]:
#Now we calculate, how much profit in an ideal case
# (forecast price = actual day-ahead price) can be made

# Load price data from CSV. The included CSV along this code is just a sample
# and should be replaced with your actual price data.
price_df = pd.read_csv(os.path.join(DATA_DIR, "price_data.csv"), index_col=0)
if "price" not in price_df.columns:
    raise ValueError("CSV must contain a column named 'price'.")

perfect_results_df = run_dispatch(price_df["price"], DEFAULT_STORAGE_PARAMS)
perfect_results_df.to_csv(os.path.join(RESULTS_DIR, "perfect_results_df_2024.csv"))

print("Dispatch results (head):")
print(perfect_results_df.head(104))
print(f"\nTotal profit: {perfect_results_df['profit'].sum()/1000:.2f}€")

In [ ]:
perfect_results_df.index = pd.to_datetime(perfect_results_df.index)

plotting.plot_price_and_dispatch(
    perfect_results_df,
    "Price, Charge Power, and Discharge Power over Time (All Timesteps)",
)

Aa we can see, the optimizer tells our battery to charge at local price-minimas and to discharge at local price-maximas. This verifies that the function works as it should.

# Preparing data for the forecast

To build a Day-Ahead forecasting model, we need DataFrames to train and test our model on. Therefore, we prepare data which we think could effect the Day-Ahead price and should be therefore given to our model. To combine all data into one DataFrame, different index-changes and adaptions have to be made. Also some time data gets quantified (like e.g. weekday, daytime or month), to also include a time-dependancy in the data.

Loading and enrichment now happen through `battery_opt.data_loading` (per-source CSV parsing) and `battery_opt.data_prep` (merging + feature engineering, shared between the 2023-2024 training set and the 2025 test set instead of being duplicated per set).

**Fixed after the review:** wind/solar/other generation features come from SMARD's day-ahead *forecasts* (`smard_client.py`) and `net_load` comes from ENTSO-E's day-ahead total load forecast (`entsoe_client.py`, document type A65) instead of realized values — realized values weren't actually available at bidding time in live operation. Both are fetched automatically into `Data/` if missing (`gen_forecast_*.csv` / `load_forecast_*.csv`); the ENTSO-E fetch needs the `ENTSOE_API_TOKEN` environment variable set (free registration at transparency.entsoe.eu) if those files aren't already cached. `res_load` still comes from realized values — no confirmed day-ahead residual-load-forecast source.

In [ ]:
#Load all data from the csv-Files from the Data folder
price_2024_df = data_loading.load_price_2024(os.path.join(DATA_DIR, "price_data.csv"))
price_2023_df = data_loading.load_price_2023(os.path.join(DATA_DIR, "price_data_2023 and others non forecasted.csv"))
price_co2_2025_df = data_loading.load_price_co2_2025(os.path.join(DATA_DIR, "price_CO2_2025.csv"))

gen_df_2023 = data_loading.load_generation(os.path.join(DATA_DIR, "gen_2023.csv"))
gen_df_2024 = data_loading.load_generation(os.path.join(DATA_DIR, "gen_2024.csv"))
gen_df_2025 = data_loading.load_generation(os.path.join(DATA_DIR, "gen_2025.csv"))

# Day-ahead generation FORECASTS (not actuals) -- fetched from SMARD if not
# already cached in Data/. This is what fixes the "actuals as features"
# feature-realism issue from the review.
smard_client.ensure_generation_forecast_data(DATA_DIR)
gen_forecast_2023 = data_loading.load_generation_forecast(smard_client.generation_forecast_path(DATA_DIR, "2023"))
gen_forecast_2024 = data_loading.load_generation_forecast(smard_client.generation_forecast_path(DATA_DIR, "2024"))
gen_forecast_2025 = data_loading.load_generation_forecast(smard_client.generation_forecast_path(DATA_DIR, "2025"))

# Day-ahead total LOAD forecast (not actuals) -- fetched from ENTSO-E if not
# already cached in Data/ (needs ENTSOE_API_TOKEN set for that first fetch).
entsoe_client.ensure_load_forecast_data(DATA_DIR)
load_forecast_2023 = data_loading.load_load_forecast(entsoe_client.load_forecast_path(DATA_DIR, "2023"))
load_forecast_2024 = data_loading.load_load_forecast(entsoe_client.load_forecast_path(DATA_DIR, "2024"))
load_forecast_2025 = data_loading.load_load_forecast(entsoe_client.load_forecast_path(DATA_DIR, "2025"))

load_df_2023 = data_loading.load_load(os.path.join(DATA_DIR, "load_2023.csv"))
load_df_2024 = data_loading.load_load(os.path.join(DATA_DIR, "load_2024.csv"))
load_df_2025 = data_loading.load_load(os.path.join(DATA_DIR, "load_2025.csv"))

co2_auction_2024_df = data_loading.load_co2_auction_2024(os.path.join(DATA_DIR, "CO2_2024.csv"))

gas_price_2023 = data_loading.load_gas_price(os.path.join(DATA_DIR, "gas_price_2023.csv"))
gas_price_2024 = data_loading.load_gas_price(os.path.join(DATA_DIR, "gas_price_2024.csv"))
gas_price_2025 = data_loading.load_gas_price(os.path.join(DATA_DIR, "gas_price_2025.csv"))

coal_23_24 = data_loading.load_coal_price(os.path.join(DATA_DIR, "coal_23_24.csv"))
coal_25 = data_loading.load_coal_price(os.path.join(DATA_DIR, "coal_25.csv"))

In [ ]:
# Gas price is reported daily -> repeat each value 24x for hourly resolution.
# Coal price is reported roughly twice a month -> average pairs into a
# "monthly" figure and broadcast that flat across every hour of the month.
gas_price_hourly_23_24_df = pd.concat(
    [data_loading.build_hourly_gas_price(gas_price_2023), data_loading.build_hourly_gas_price(gas_price_2024)],
    ignore_index=True,
)
gas_price_hourly_25_df = data_loading.build_hourly_gas_price(gas_price_2025)

coal_price_hourly_24_df = data_loading.build_hourly_coal_price(coal_23_24, start_year=2023)
coal_price_hourly_25_df = data_loading.build_hourly_coal_price(coal_25, start_year=2023)

We merge all data (of 2023 and 2024) into one DataFrame with which we want to test, train and validate our model. We also create a new DataFrame with the same data for the first half of 2025. With this dataframe, we want to analyse, how good our forecast is working.

In [ ]:
#Create the training DataFrame (2023+2024) and the 2025 testrun DataFrame
gen_forecast_23_24_df = pd.concat([gen_forecast_2023, gen_forecast_2024], ignore_index=True)
load_forecast_23_24_df = pd.concat([load_forecast_2023, load_forecast_2024], ignore_index=True)

forecasting_data, testrun_df = data_prep.build_feature_dataset(
    price_2023_df, price_2024_df, co2_auction_2024_df, price_co2_2025_df,
    gen_df_2023, gen_df_2024, gen_df_2025,
    load_df_2023, load_df_2024, load_df_2025,
    gas_price_hourly_23_24_df, gas_price_hourly_25_df,
    coal_price_hourly_24_df, coal_price_hourly_25_df,
    gen_forecast_23_24_df, gen_forecast_2025,
    load_forecast_23_24_df, load_forecast_2025,
)

#Safe forecasting and testrun to .csv and store in sample_data folder
forecasting_data.to_csv("forecasting_data.csv")
testrun_df.to_csv("testrun_df.csv")

#display forecasting_data (examplary part of it)
#display(forecasting_data)
#display(testrun_df)
#forecasting_data.head(25)

# Implement the random forest model

Now, we are going to implement our random forest model. As our test-data, we use all data from 2023 and 2024. We let our test run for the data of the first half of 2025. We create our forecast-model using the training-data and the randomforest regressor.

First, we split our forecasting_data into train and validation. We use the validation-data to tune our hyperparameteres and the train-data to train the RandomForest. To test, we then use later the data for 2025.

In [ ]:
# ===========================================
# Random forest model
# ===========================================

features, target = modeling.prepare_features_target(forecasting_data)

# split our data into train and validation (shuffle=False: don't shuffle due to data-leakage)
X_train, X_val, y_train, y_val = modeling.train_val_split(features, target)

print(f"Train set: {X_train.shape}, Validation set: {X_val.shape}")

In [ ]:
rf_model, y_pred_train_rf, y_pred_val_rf, metrics = modeling.train_random_forest(
    X_train, y_train, X_val, y_val
)

print('Training MAE:', metrics["mae_train"])
print('Validation MAE:', metrics["mae_val"])
print('Training RMSE:', metrics["rmse_train"])
print('Validation RMSE:', metrics["rmse_val"])

In [ ]:
# Hyperparameter Tuning
grid_rf, best_rf, tuned_mae = modeling.tune_random_forest(X_train, y_train, X_val, y_val)

print('Best RF Params:', grid_rf.best_params_)
print('Tuned RF MAE:', tuned_mae)

To increase the quality of our prediction, we tune the hyperparameters (number of estimators and depth of each tree) via `TimeSeriesSplit` cross-validation (fixed after the review: plain K-fold CV would let future rows help predict past ones within a fold). The main model above already uses n_estimators=180, max_depth=10 from an earlier such run; re-running the grid search may find slightly different values since it doesn't feed back into the deployed model automatically.

In [ ]:
# Plotting importance of different features

#Load the persisted RandomForest model
loaded_rf = modeling.load_random_forest()

importances = modeling.feature_importances(loaded_rf, X_train.columns)
plotting.plot_feature_importance(importances)

Now we also want to plot the forecasted against the predicted Day-Ahead price to become a feeling on how close these values are to each other.

In [ ]:
# ============================================
# Time Series Comparison: Predicted vs. Actual
# ============================================
plotting.plot_prediction_vs_actual(
    y_val.values, y_pred_val_rf,
    "Validation Set Predictions of Day Ahead Price over Time",
    xlabel="Time (Hour))",
    ylim=(-70, 1000),
)

In [ ]:
# Plotting the first 168 time steps of the test set predictions vs actual
plotting.plot_prediction_vs_actual(
    y_val.values, y_pred_val_rf,
    "Validation-Set Prediction vs. Acutal of Day Ahead Price over one week",
    xlabel="Hours (of one week)",
    n_steps=168,
    ylim=(-70, 250),
)

# Forecasting 2025

Now we want to analyze, how well our forecasting model works and how much profit we could have generated in the first half of 2025. Therefore we first calculate, how much profit our optimizer generates if our forecast would work perfectly. Therefore, we use the actual day-ahead prices of 2025 and feed those prices into our optimazor. The result is the maximum profit, we could have generated with the battery, defined in the optimizor.

In [ ]:
#Now we calculate, how much profit in an ideal case can be made in 2025
# (forecast price = actual day-ahead price) can be made
price_df_2025 = pd.read_csv("testrun_df.csv", index_col=4)
if "price" not in price_df_2025.columns:
    raise ValueError("CSV must contain a column named 'price'.")

perfect_results_df_2025 = run_dispatch(price_df_2025["price"], DEFAULT_STORAGE_PARAMS)
perfect_results_df_2025.to_csv(os.path.join(RESULTS_DIR, "perfect_results_df_2025.csv"))

average_price_2025 = price_df_2025["price"].mean()
theoretical_total_profit = perfect_results_df_2025["profit"].sum() / 1000

print("Dispatch results (head):")
print(perfect_results_df_2025.head(104))
print(f"\nTheoretical profit: {theoretical_total_profit:.2f} €")
print(f"\nAverage price 2025: {average_price_2025:.2f} €/MWh")

Similar as before, we wat to visualize the behavior of the optimizor in relation to the price.

In [ ]:
# Re-read with the datetime index parsed, for plotting; this is the version
# of perfect_results_df_2025 used as the "ground truth" reference from here on.
perfect_results_df_2025 = backtest.load_dispatch_csv(
    os.path.join(RESULTS_DIR, "perfect_results_df_2025.csv"), date_format="%d.%m.%Y %H:%M"
)

plotting.plot_price_and_dispatch(
    perfect_results_df_2025,
    "Price, Charge Power and Discharge Power over Time (Actual Dad-Ahead Price)",
    price_label="Day-Ahead Price (€/MWh)",
)

Now want wo check, how well our forecast can be used to run the optimizor and to generate profits. Therefore, we first predict the Day-Ahead prices of 2025 using our trained forecast model.
We feed it with the 2025 data (without the given DA-prices) and store the resulting price formation in the Results folder.

In [ ]:
# Select features from testrun_df for forecasting (exclude 'price' and 'time')
# Ensure the columns match the training data (X_train) in name and order
forecasted_prices_df = modeling.forecast_prices(loaded_rf, testrun_df, features.columns)

#Save the forecasted_prices as a .csv in the Results folder
forecasted_prices_df.to_csv(os.path.join(RESULTS_DIR, "forecasted_prices_2025.csv"))

# Display the forecasted prices
#print("Forecasted Prices (head):")
#display(forecasted_prices_df.head())

In [ ]:
# Calculate the MAE/RMSE for our test data
rf_mae, rf_rmse = modeling.evaluate(perfect_results_df_2025["price"], forecasted_prices_df["forecasted_price"])

print(f"Random Forest MAE: {rf_mae:.2f} €/MWh")
print(f"Random Forest RMSE: {rf_rmse:.2f} €/MWh")

In [ ]:
# Plotting the first 169 time steps of the test set predictions vs actual
plotting.plot_prediction_vs_actual(
    testrun_df["price"].values, forecasted_prices_df["forecasted_price"].values,
    "Test-Set Prediction vs. Actual Day Ahead Price of 2025 over one week",
    xlabel="Hours (of one week)",
    n_steps=169,
    ylim=(-70, 250),
)

Now we want to calculate, how much profit we can theoredically make, when we use our predicition to optimize our battery, while concidering the predicted Day-Ahead prices are the actual prices (so we assume, the predicted prices of our model are the real ones). We also plot a comparison of the predicted and actual DA-prices. We also plot and save the resulting data in this case.

In [ ]:
#Now we calculate, how much profit we can make, using our predicted prices
# (forecast price = handled as actual day-ahead price) can be made
forecasted_results_df_2025 = backtest.run_and_save_dispatch(
    forecasted_prices_df["forecasted_price"], "forecasted_results_df_2025.csv"
)

In [ ]:
# Re-read with the datetime index parsed, for plotting
forecasted_results_2025_plot = backtest.load_dispatch_csv(
    os.path.join(RESULTS_DIR, "forecasted_results_df_2025.csv"), date_format="%d.%m.%Y %H:%M"
)

plotting.plot_price_and_dispatch(
    forecasted_results_2025_plot,
    "Price, Charge Power and Discharge Power over Time (Forecasted Day-Ahead Price)",
    price_label="Day-Ahead Price (€/MWh)",
)

Finally, we calculate our "real" profit, which creates itself out of the charge and discharge behavior from our Day-Ahead forecast and the actual Day-Ahead prices. This information helps us calculating the actual difference in profit, related to our forecasting model.

In [ ]:
#Create a new dataframe with P_ch, P_dis and E from forecasted prices, but
#re-priced against the actual Day-Ahead price (perfect_results_df_2025)
actual_results_df_2025 = backtest.compute_actual_profit(forecasted_results_df_2025, perfect_results_df_2025)
actual_results_df_2025.to_csv(os.path.join(RESULTS_DIR, "actual_results_df_2025.csv"))

actual_generated_profit = actual_results_df_2025["profit"].sum() / 1000

print("Dispatch results (head):")
print(actual_results_df_2025.head(104))
print(f"\nActual profit (forecasted charge/discharge and actual prices): {actual_generated_profit:.2f} €")

In [ ]:
plotting.plot_price_and_dispatch(
    actual_results_df_2025,
    "Actual and Forecasted Price, Charge Power and Discharge Power over Time (Forecasted utilization, actual price)",
    price_label="Day-Ahead Price (€/MWh)",
    forecasted_price=forecasted_results_2025_plot["price"],
)

### Battery signal report

One combined table of how the battery would actually be operated off our forecast signal: the charge/discharge/state-of-charge schedule the optimizer chose from the *predicted* price, next to the *predicted* price, the *actual* price, and the resulting real profit per hour.

In [ ]:
battery_signal_report_2025 = backtest.build_signal_report(
    forecasted_results_df_2025, forecasted_prices_df["forecasted_price"], perfect_results_df_2025
)
battery_signal_report_2025.to_csv(os.path.join(RESULTS_DIR, "battery_signal_report_2025.csv"))

battery_signal_report_2025.head(24)

In [ ]:
battery_cumulative_profit_2025 = backtest.cumulative_profit_by_day(battery_signal_report_2025)
battery_cumulative_profit_2025.to_csv(os.path.join(RESULTS_DIR, "battery_cumulative_profit_2025.csv"))

battery_cumulative_profit_2025.head()

# Sensitivity Analysis

As final task in this work, we want to do a sensitivity analysis of our forecast, so we can detect, how the actual profit is effected by the forecast. Therefore, we performed two sensitivity analysis:

1.   We changed the MAE of our forecast by +-1 Euro/MWh
2.   We changed the MAE of our forecast by +-10%

Therefore, we took the price of each timestep of our forecast, compared it with the actual price and made the error smaller (+-1 Euro/MWH or +-10%)

The correction functions themselves (`correct_price_toward_actual_abs`, `correct_price_away_from_actual_abs`, `correct_price_fraction`) live in `battery_opt.backtest` and are reused for both sensitivities below and for the extended sweep at the end.

## 1. Sensitivity Analysis (+-1 Euro/MWh)

Here we adapt the forecast prices in comparison to the actual prices (+-1 Euro/MWh). Then we added the adapted prices to a dataframe. There prices are then used to calculate a charge/discharge behavior. Finally, this behavior gets combined with the actual prices and a profit gets calculated by the optimizer.

In [ ]:
# Adjusting our predicted price by +-1€/MWh closer to / further from the real price
forecasted_prices_corrected_df = forecasted_prices_df.copy()
forecasted_prices_corrected_df.index = perfect_results_df_2025.index
forecasted_prices_corrected_df["real_price"] = perfect_results_df_2025["price"]

forecasted_prices_corrected_df["corrected_price_positive"] = backtest.correct_price_toward_actual_abs(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"]
)
forecasted_prices_corrected_df["corrected_price_negative"] = backtest.correct_price_away_from_actual_abs(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"]
)

forecasted_prices_corrected_df.to_csv(
    os.path.join(RESULTS_DIR, "forecasted_price_2025_corrected_1_Euro_MWh.csv")
)
forecasted_prices_corrected_df.head()

In [ ]:
# Plotting 169 timesteps to see how the sensitivity behaves in comparison with the actual price
plotting.plot_sensitivity_correction(
    forecasted_prices_corrected_df["real_price"].values,
    forecasted_prices_corrected_df["forecasted_price"].values,
    forecasted_prices_corrected_df["corrected_price_positive"].values,
    forecasted_prices_corrected_df["corrected_price_negative"].values,
    "Sensitivity Analysis (+-1Euro/MWh) Day Ahead Price of 2025 over Time (First 100 Timesteps)",
)

In [ ]:
# Now we run the optimizer with the corrected prices and see how much profit
# we can get and how P_ch / P_dis behave
dispatch_positive_1eur = run_dispatch(forecasted_prices_corrected_df["corrected_price_positive"], DEFAULT_STORAGE_PARAMS)
dispatch_negative_1eur = run_dispatch(forecasted_prices_corrected_df["corrected_price_negative"], DEFAULT_STORAGE_PARAMS)

We calculate the actual profit we would make with our positive- and negative-tuned forecast (-1 / +1 Euro/MWh MAE)

In [ ]:
actual_positive_1eur = backtest.compute_actual_profit(dispatch_positive_1eur, perfect_results_df_2025)
actual_negative_1eur = backtest.compute_actual_profit(dispatch_negative_1eur, perfect_results_df_2025)

profit_positive_1eur = actual_positive_1eur["profit"].sum() / 1000
profit_negative_1eur = actual_negative_1eur["profit"].sum() / 1000

print(f"Total profit with positive tuned price (-1 Euro/MWh): {profit_positive_1eur:.2f} €")
print(f"Total profit with negative tuned price (+1 Euro/MWh): {profit_negative_1eur:.2f} €")

## 2. Sensitivity analysis (+-10% RMSE)

Since the increase/decrease of the MAE has only a very low impact, we want to change the error even more. Therefore, we always assume a 10% decrease/increase of the error for every timestep.

In [ ]:
forecasted_prices_corrected_df["corrected_price_positive_10_percent"] = backtest.correct_price_fraction(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"],
    fraction=0.1, toward_actual=True,
)
forecasted_prices_corrected_df["corrected_price_negative_10_percent"] = backtest.correct_price_fraction(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"],
    fraction=0.1, toward_actual=False,
)

forecasted_prices_corrected_df.to_csv(
    os.path.join(RESULTS_DIR, "forecasted_price_2025_corrected_10_percent.csv")
)

In [ ]:
plotting.plot_sensitivity_correction(
    forecasted_prices_corrected_df["real_price"].values,
    forecasted_prices_corrected_df["forecasted_price"].values,
    forecasted_prices_corrected_df["corrected_price_positive_10_percent"].values,
    forecasted_prices_corrected_df["corrected_price_negative_10_percent"].values,
    "Sensitivity Analysis (10%) Day Ahead Price of 2025 over Time (First 100 Timesteps)",
)

In [ ]:
dispatch_positive_10pct = run_dispatch(forecasted_prices_corrected_df["corrected_price_positive_10_percent"], DEFAULT_STORAGE_PARAMS)
dispatch_negative_10pct = run_dispatch(forecasted_prices_corrected_df["corrected_price_negative_10_percent"], DEFAULT_STORAGE_PARAMS)

actual_positive_10pct = backtest.compute_actual_profit(dispatch_positive_10pct, perfect_results_df_2025)
actual_negative_10pct = backtest.compute_actual_profit(dispatch_negative_10pct, perfect_results_df_2025)

corrected_profit_positive_10_percent = actual_positive_10pct["profit"].sum() / 1000
corrected_profit_negative_10_percent = actual_negative_10pct["profit"].sum() / 1000

print(f"Total profit with positive tuned price (-10% MAE): {corrected_profit_positive_10_percent:.2f} €")
print(f"Total profit with negative tuned price (+10% MAE): {corrected_profit_negative_10_percent:.2f} €")

In [ ]:
# Checking how much profit has changed
change_positive = corrected_profit_positive_10_percent / actual_generated_profit
change_negative = corrected_profit_negative_10_percent / actual_generated_profit

print('Compared to actual profit:')
print(f"\nIncrease of Profit: {(change_positive-1)*100:.2f} %")
print(f"\nDecrease of Profit: {(change_negative-1)*100:.2f} %")

print('----------------------------------------------------------------------')
print('Compared to maximum profit:')
print(f"\nDecrease Profit for Forecasted Prices:  {actual_generated_profit/theoretical_total_profit*100-100:.2f} %")
print(f"\nDecrease Profit for -10% RMSE: {corrected_profit_positive_10_percent/theoretical_total_profit*100-100:.2f} %")
print(f"\nDecrease Profit for +10% RMSE: {corrected_profit_negative_10_percent/theoretical_total_profit*100-100:.2f} %")

As a small extra, we wanted to see, how the performance of our forecasting in combination with the optimizer effects the payback period of the assumed battery system.

Therefore we assume, that a battery system costs roundabout 250 Euro/kWh, refering to a "turnkey plant" (source: ENBW). For simplification, we don't have any operating- or maintaining costs. First, we therefore calculate the cost of the total 50 MWh plant. Then we look, how much of the cost of this plant can be covered with the different profits (perfect szenario, actual profit and +-10% RSME) in the first half of 2025.

Then we assume, that for the future, the performance of the optimizer stays the same and look, after how many years our system payed itself back. For simplification, we concider the average Day-Ahead price similar to the one of the first half of 2025.

In [ ]:
#Calculate cost of battery system and payback period per scenario
print(f"\nAverage Day-Ahead price first half of 2025: {average_price_2025:.2f} Euro/MWh")
print(f"\nCost for the whole battery system: {BATTERY_SYSTEM_COST_EUR:.0f} Euro")

scenarios = {
    "Perfect price forecast": theoretical_total_profit,
    "Actual price forecast": actual_generated_profit,
    "Actual price forecast -10% RMSE": corrected_profit_positive_10_percent,
    "Actual price forecast +10% RMSE": corrected_profit_negative_10_percent,
}

for name, profit in scenarios.items():
    percent_paid_back, years = backtest.payback_years(profit, BATTERY_SYSTEM_COST_EUR)
    print("-----------------------------------------------------------------------")
    print(f"{name}:")
    print(f"\nPercentage of plant payed back after the first half of 2025: {percent_paid_back*100:.2f}%")
    print(f"\nYears needed to pay back the plant: {years:.2f} years")

In [ ]:
# Calculate the percentage of theoretical maximum profit for each scenario
actual_profit_percent = backtest.profit_percent_of_theoretical(actual_generated_profit, theoretical_total_profit)
corrected_positive_profit_percent = backtest.profit_percent_of_theoretical(corrected_profit_positive_10_percent, theoretical_total_profit)
corrected_negative_profit_percent = backtest.profit_percent_of_theoretical(corrected_profit_negative_10_percent, theoretical_total_profit)

plotting.plot_profit_comparison(
    ["Theoretical Profit", "Actual Profit", "-10% MAE Profit", "+10% MAE Profit"],
    [100, actual_profit_percent, corrected_positive_profit_percent, corrected_negative_profit_percent],
    ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"],
)

In [ ]:
plotting.plot_profit_comparison(
    ["Theoretical Profit", "Actual Profit"],
    [100, actual_profit_percent],
    ["#1f77b4", "#2ca02c"],
    title="Comparison of Thoeretical and Actual Generated Profit",
)

#Extra

Finally, we wanted to investigate the behavior of the price, when we adapt the RMSE more dramatically. Therefore we change it from 10%-90% in positive and negative direction, run the optimizer for each step and calculate the profit. We plot both the results for a increasing and decreasing RMSE.

FYI: This last part of the code was written out of curiosity. We weren't able to investigate it appropriately. We were not able to clean it and therefore it could be a bit messy. We will show it in the presentation as small extra in the end.

(Reworked here to reuse `backtest.run_sensitivity_sweep` instead of writing/reading a CSV per sweep step.)

In [ ]:
fractions = [f / 100 for f in range(10, 91, 10)]

profits_positive = backtest.run_sensitivity_sweep(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"],
    fractions, toward_actual=True,
)
profits_negative = backtest.run_sensitivity_sweep(
    forecasted_prices_corrected_df["forecasted_price"], forecasted_prices_corrected_df["real_price"],
    fractions, toward_actual=False,
)

print(list(profits_positive.values()))
print(list(profits_negative.values()))

In [ ]:
# Calculate the percentage difference from the actual profit for each profit in the list
percentage_differences_positive = [
    (profit - actual_generated_profit) / actual_generated_profit * 100 for profit in profits_positive.values()
]
rmse_percentage_changes = [int(f * 100) for f in fractions]

plotting.plot_profit_delta_vs_error(rmse_percentage_changes, percentage_differences_positive, "Reduction of RMSE in (%)")

In [ ]:
percentage_differences_negative = [
    (profit - actual_generated_profit) / actual_generated_profit * 100 for profit in profits_negative.values()
]

plotting.plot_profit_delta_vs_error(rmse_percentage_changes, percentage_differences_negative, "Increase of RMSE in (%)")